In [2]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from typing import Tuple, Optional, Dict, List

# ==========================================
# 1. الدالة الأساسية: حساب زاوية المفصل (Joint Flexion Angle)
# ==========================================
# تطبق المعادلة من القسم 3.3.3 في التقرير:
# θ = arccos((u⃗ · v⃗) / (||u⃗|| × ||v⃗||)) × (180°/π)
# حيث u⃗ = A - B⃗   و   v⃗ = C - B⃗

def calculate_joint_angle(
    A: Tuple[float, float],  # المفصل القريب (Proximal)
    B: Tuple[float, float],  # المفصل الرأسي (Vertex) - عندها تُقاس الزاوية
    C: Tuple[float, float]   # المفصل البعيد (Distal)
) -> Optional[float]:
    """
    تحسب الزاوية عند المفصل B (بالدرجات) باستخدام قانون جيب التمام.
    تعيد None إذا كانت أي نقطة فارغة أو إذا كان الطول صفراً.
    """
    if A is None or B is None or C is None:
        return None
    
    # تحويل النقاط إلى مصفوفات NumPy
    A_vec = np.array(A, dtype=float)
    B_vec = np.array(B, dtype=float)
    C_vec = np.array(C, dtype=float)
    
    # حساب المتجهين u⃗ و v⃗
    u = A_vec - B_vec
    v = C_vec - B_vec
    
    # حساب الأطوال (||u⃗|| و ||v⃗||)
    norm_u = np.linalg.norm(u)
    norm_v = np.linalg.norm(v)
    
    # تجنب القسمة على صفر
    if norm_u < 1e-6 or norm_v < 1e-6:
        return None
    
    # حساب جيب تمام الزاوية: cos(θ) = (u⃗ · v⃗) / (||u⃗|| × ||v⃗||)
    cos_theta = np.dot(u, v) / (norm_u * norm_v)
    
    # قص القيمة إلى النطاق [-1, 1] لتجنب NaN بسبب أخطاء التقريب العائم
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    
    # تحويل إلى درجات
    angle_rad = np.arccos(cos_theta)
    angle_deg = np.degrees(angle_rad)
    
    return float(angle_deg)


# ==========================================
# 2. دالة استخراج إحداثيات نقطة واحدة بأمان
# ==========================================
def get_landmark_coords(
    landmarks, 
    landmark_id: int, 
    visibility_threshold: float = 0.5
) -> Optional[Tuple[float, float]]:
    """
    ترجع إحداثيات (x, y) لنقطة معينة إذا كانت درجة الثقة >= العتبة.
    الإحداثيات نسبية (0.0 - 1.0) - الزوايا لا تتأثر بمقياس الصورة.
    """
    lm = landmarks.landmark[landmark_id]
    if lm.visibility < visibility_threshold:
        return None
    return (lm.x, lm.y)


# ==========================================
# 3. دالة حساب مؤشر التماثل (Robinson Symmetry Index)
# ==========================================
# تطبق المعادلة من القسم 3.3.3:
# SI = (2 × |XLeft - XRight|) / (XLeft + XRight) × 100%
# القيم > 10% تشير إلى انحراف غير متماثل ملحوظ

def calculate_symmetry_index(left_val: float, right_val: float) -> Optional[float]:
    """
    تحسب مؤشر روبنسون للتماثل بين الجانبين.
    تعيد None إذا كان المجموع صفراً أو إذا كانت أي قيمة مفقودة.
    """
    if left_val is None or right_val is None:
        return None
    
    denominator = left_val + right_val
    if denominator < 1e-6:
        return None
    
    si = (2.0 * abs(left_val - right_val)) / denominator * 100.0
    return float(si)


# ==========================================
# 4. دالة استخراج كل خصائص إطار واحد
# ==========================================
# معرّف MediaPipe IDs (من التوثيق الرسمي):
# Hip:     23 (Left), 24 (Right)
# Knee:    25 (Left), 26 (Right)
# Ankle:   27 (Left), 28 (Right)
# Shoulder:11 (Left), 12 (Right)
# Heel:    29 (Left), 30 (Right)

def extract_frame_features(landmarks) -> Dict[str, Optional[float]]:
    """
    تستخرج كل الزوايا الحيوية ومؤشر التماثل لإطار واحد.
    """
    features = {}
    
    if landmarks is None:
        # إرجاع None لكل خاصية إذا لم يُكتشف أي هيكل
        for name in ["L_Hip", "L_Knee", "L_Ankle", 
                     "R_Hip", "R_Knee", "R_Ankle",
                     "Knee_Symmetry", "Hip_Symmetry", "Ankle_Symmetry"]:
            features[name] = None
        return features
    
    # --- استخراج الإحداثيات ---
    L_shoulder = get_landmark_coords(landmarks, 11)
    R_shoulder = get_landmark_coords(landmarks, 12)
    L_hip      = get_landmark_coords(landmarks, 23)
    R_hip      = get_landmark_coords(landmarks, 24)
    L_knee     = get_landmark_coords(landmarks, 25)
    R_knee     = get_landmark_coords(landmarks, 26)
    L_ankle    = get_landmark_coords(landmarks, 27)
    R_ankle    = get_landmark_coords(landmarks, 28)
    L_heel     = get_landmark_coords(landmarks, 29)
    R_heel     = get_landmark_coords(landmarks, 30)
    
    # --- حساب زوايا الركبة ---
    # Knee Angle: Hip (A) - Knee (B) - Ankle (C)
    features["L_Knee"] = calculate_joint_angle(L_hip, L_knee, L_ankle)
    features["R_Knee"] = calculate_joint_angle(R_hip, R_knee, R_ankle)
    
    # --- حساب زوايا الورك ---
    # Hip Angle: Shoulder (A) - Hip (B) - Knee (C)
    features["L_Hip"] = calculate_joint_angle(L_shoulder, L_hip, L_knee)
    features["R_Hip"] = calculate_joint_angle(R_shoulder, R_hip, R_knee)
    
    # --- حساب زوايا الكاحل ---
    # Ankle Angle: Knee (A) - Ankle (B) - Heel (C)
    features["L_Ankle"] = calculate_joint_angle(L_knee, L_ankle, L_heel)
    features["R_Ankle"] = calculate_joint_angle(R_knee, R_ankle, R_heel)
    
    # --- حساب مؤشرات التماثل (Robinson Index) ---
    features["Knee_Symmetry"] = calculate_symmetry_index(
        features["L_Knee"], features["R_Knee"]
    )
    features["Hip_Symmetry"] = calculate_symmetry_index(
        features["L_Hip"], features["R_Hip"]
    )
    features["Ankle_Symmetry"] = calculate_symmetry_index(
        features["L_Ankle"], features["R_Ankle"]
    )
    
    return features


# ==========================================
# 5. خط الأنابيب الكامل: من الفيديو إلى DataFrame
# ==========================================
def process_video_to_features(
    video_path: str,
    visibility_threshold: float = 0.5
) -> pd.DataFrame:
    """
    يقرأ الفيديو إطاراً بإطار، ويستخرج كل الخصائص الحيوية،
    ويعيد DataFrame منظمًا جاهزاً للتحليل أو التدريب.
    """
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(
        static_image_mode=False,
        model_complexity=1,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"❌ تعذر فتح الفيديو: {video_path}")
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    all_features: List[Dict] = []
    frame_idx = 0
    
    print(f"🎬 بدأ استخراج الخصائص: {total_frames} إطار @ {fps:.1f} FPS")
    
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break
        
        # تحويل إلى RGB لـ MediaPipe
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(frame_rgb)
        
        # استخراج الخصائص للإطار الحالي
        frame_features = extract_frame_features(
            results.pose_landmarks
        )
        
        # إضافة معلومات الإطار
        frame_features["Frame"] = frame_idx
        frame_features["Timestamp_sec"] = frame_idx / fps if fps > 0 else 0.0
        
        all_features.append(frame_features)
        frame_idx += 1
        
        if frame_idx % 30 == 0:
            print(f"⏳ تقدّم المعالجة: {frame_idx}/{total_frames}")
    
    cap.release()
    pose.close()
    
    # تحويل إلى DataFrame وإعادة ترتيب الأعمدة
    df = pd.DataFrame(all_features)
    cols = ["Frame", "Timestamp_sec",
            "L_Hip", "L_Knee", "L_Ankle",
            "R_Hip", "R_Knee", "R_Ankle",
            "Knee_Symmetry", "Hip_Symmetry", "Ankle_Symmetry"]
    df = df[cols]
    
    print(f"✅ تم استخراج {len(df)} صف بنجاح.")
    return df


# ==========================================
# 6. نقطة التشغيل الرئيسية (Main)
# ==========================================
if __name__ == "__main__":
    VIDEO_PATH = r"C:\Users\hacker\Desktop\machine_learnig\Gaitly_AI\viduo\WhatsApp Video 2026-10-17 at 1.13.06 AM.mp4"   # نفس الفيديو من المهمة 2
    OUTPUT_CSV = "gait_features.csv"
    
    # تشغيل خط الأنابيب
    features_df = process_video_to_features(VIDEO_PATH)
    
    # حفظ النتائج
    features_df.to_csv(OUTPUT_CSV, index=False)
    print(f"💾 تم حفظ الخصائص في: {OUTPUT_CSV}")
    
    # عرض ملخص إحصائي سريع
    print("\n📊 ملخص إحصائي للزوايا (بالدرجات):")
    print(features_df.describe().round(2))
    
    # تحليل مؤشر التماثل
    print("\n⚖️ تحليل التماثل (Robinson Index):")
    for col in ["Knee_Symmetry", "Hip_Symmetry", "Ankle_Symmetry"]:
        valid = features_df[col].dropna()
        if len(valid) > 0:
            mean_si = valid.mean()
            status = "⚠️ غير متماثل (>10%)" if mean_si > 10 else "✅ متماثل"
            print(f"  {col}: متوسط = {mean_si:.2f}%  {status}")

🎬 بدأ استخراج الخصائص: 433 إطار @ 30.0 FPS
⏳ تقدّم المعالجة: 30/433
⏳ تقدّم المعالجة: 60/433
⏳ تقدّم المعالجة: 90/433
⏳ تقدّم المعالجة: 120/433
⏳ تقدّم المعالجة: 150/433
⏳ تقدّم المعالجة: 180/433
⏳ تقدّم المعالجة: 210/433
⏳ تقدّم المعالجة: 240/433
⏳ تقدّم المعالجة: 270/433
⏳ تقدّم المعالجة: 300/433
⏳ تقدّم المعالجة: 330/433
⏳ تقدّم المعالجة: 360/433
⏳ تقدّم المعالجة: 390/433
⏳ تقدّم المعالجة: 420/433
✅ تم استخراج 433 صف بنجاح.
💾 تم حفظ الخصائص في: gait_features.csv

📊 ملخص إحصائي للزوايا (بالدرجات):
        Frame  Timestamp_sec   L_Hip  L_Knee  L_Ankle   R_Hip  R_Knee  \
count  433.00         433.00  303.00  292.00   292.00  303.00  293.00   
mean   216.00           7.20  142.43  157.47   124.21  149.81  140.01   
std    125.14           4.17   17.55   24.68    20.34   16.56   28.18   
min      0.00           0.00   48.51   11.42    33.56   43.06   53.00   
25%    108.00           3.60  134.85  149.57   113.28  142.13  125.68   
50%    216.00           7.20  144.74  165.51   123.19  15

In [3]:
# اختبار: زاوية قائمة (90°) بين متجهين متعامدين
angle = calculate_joint_angle(A=(1, 0), B=(0, 0), C=(0, 1))
print(f"زاوية قائمة محسوبة: {angle:.2f}°")  # يجب أن تطبع 90.00°

# اختبار: زاوية مستقيمة (180°)
angle = calculate_joint_angle(A=(-1, 0), B=(0, 0), C=(1, 0))
print(f"زاوية مستقيمة محسوبة: {angle:.2f}°")  # يجب أن تطبع 180.00°

زاوية قائمة محسوبة: 90.00°
زاوية مستقيمة محسوبة: 180.00°
